In [ ]:
# Cell 00 (E046 frozen-input candidate): submit Version 5's settings (Humming dense, CUDA graphs <= 64, max-num-seqs
# 64, util 0.945; Kaggle KV 1.55M) + one rollout per task for tasks with several test inputs (ARC_MULTI_TEST) with a
# (1 + 0.5 (n - 1)) x output budget / turn cap (ARC_MULTI_BUDGET); 64 workers, KV gate 0.85.
# E128 on the 45 multi-pair eval tasks (E046 run 3): pass@2 83.1% vs 77.5% per pair at -29% tokens per task.
# ARC_CONTEXT_LIMIT = the server's max-model-len: turns never request past it (a 3-input task overflowed in run 3).
import json, os
for _k, _v in (("ARC_WORKERS", "64"),
               ("ARC_MULTI_TEST", "1"),
               ("ARC_MULTI_BUDGET", "0.5"),
               ("ARC_CONTEXT_LIMIT", "163840"),
               ("ARC_MAX_NUM_SEQS", "64"),
               ("ARC_GPU_UTIL", "0.945"),
               ("ARC_KV_ADMIT_MAX", "0.85"),
               ("ARC_KV_ADMIT_MIN_WORKERS", "16"),
               ("ARC_AUTO_REPAIR", "1"),
               ("ARC_ADMIT_MAX_WAITING", "1"),
               ("ARC_SERVER_EXTRA_ARGS", json.dumps(["--linear-backend", "humming", "--compilation-config",
                                                     json.dumps({"cudagraph_mode": "FULL",
                                                                 "max_cudagraph_capture_size": 64})]
                                                    + ["--prefix-cache-retention-interval", "512", "--kv-transfer-config", "{\"kv_connector\":\"OffloadingConnector\",\"kv_role\":\"kv_both\",\"kv_connector_extra_config\":{\"cpu_bytes_to_use\":34359738368,\"offload_prompt_only\":false,\"swa_store_policy\":\"checkpoint\",\"shared_region\":false,\"eviction_policy\":\"lru\"}}"]))):
    os.environ.setdefault(_k, _v)
print('PYTORCH_CUDA_ALLOC_CONF from the image:', repr(os.environ.get('PYTORCH_CUDA_ALLOC_CONF')), flush=True)
_ac = [x for x in os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '').split(',') if x and 'expandable_segments' not in x]
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ','.join(_ac)
# The commit run ("Save & Run All") is a smoke test: server start + 3 tasks (2 multi-test). The competition rerun
# ignores ARC_COMMIT_MODE (cell 0 always runs "submit") and never sees the task list.
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    os.environ.setdefault("ARC_COMMIT_MODE", "smoke")
    os.environ.setdefault("ARC_TASK_IDS", "25ff71a9,27a28665,00576224")
    # smoke within 30 min of the first cell: model load, generation, force cut and final write all run
    os.environ.setdefault("ARC_STOP_NEW_S", "1500")
    os.environ.setdefault("ARC_FORCE_AT_S", "1620")
    os.environ.setdefault("ARC_HARD_STOP_S", "1680")
    os.environ.setdefault("ARC_KILL_AT_S", "1770")
    import shutil, subprocess, threading, time as _st
    os.environ.setdefault("ARC_SMOKE_ENV_LOG", "/kaggle/working/smoke_env.jsonl")

    def _smoke_env_sampler(path=os.environ["ARC_SMOKE_ENV_LOG"]):
        def _rd(p):
            try:
                return open(p).read().strip()
            except Exception:
                return None
        while True:
            try:
                rec = {"t": round(_st.time(), 1)}
                for k in ("memory.current", "memory.peak", "memory.max", "memory.swap.current"):
                    rec[k] = _rd(f"/sys/fs/cgroup/{k}")
                rec["stat"] = {l.split()[0]: int(l.split()[1]) for l in (_rd("/sys/fs/cgroup/memory.stat") or "").splitlines()
                               if l.split()[:1] and l.split()[0] in ("anon", "file", "shmem", "unevictable", "file_mapped")}
                rec["meminfo_kb"] = {l.split(":")[0]: int(l.split()[1]) for l in (_rd("/proc/meminfo") or "").splitlines()
                                     if l.split(":")[0] in ("MemTotal", "MemAvailable", "Shmem", "Mlocked", "Cached")}
                try:
                    rec["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=index,power.draw,power.limit,clocks.sm,"
                                                 "memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"],
                                                capture_output=True, text=True, timeout=20).stdout.strip().splitlines()
                except Exception:
                    rec["gpu"] = []
                rec["disk_free_gib"] = {d: round(shutil.disk_usage(d).free / 2**30, 1)
                                        for d in ("/kaggle/working", "/kaggle/temp", "/kaggle/tmp", "/tmp", "/dev/shm")
                                        if os.path.exists(d)}
                with open(path, "a") as f:
                    f.write(json.dumps(rec) + "\n")
            except Exception:
                pass
            _st.sleep(30)
    threading.Thread(target=_smoke_env_sampler, daemon=True).start()
print("E046 settings:", {k: os.environ.get(k) for k in ("ARC_WORKERS", "ARC_MULTI_TEST", "ARC_MULTI_BUDGET",
                                                     "ARC_CONTEXT_LIMIT", "ARC_KV_ADMIT_MAX", "ARC_MAX_NUM_SEQS", "ARC_GPU_UTIL",
                                                     "ARC_REASONING_EFFORT", "ARC_AUTO_REPAIR", "ARC_BEST_GUESS_AT",
                                                     "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_ADMIT_MAX_WAITING",
                                                     "ARC_STOP_NEW_S", "ARC_FORCE_AT_S", "ARC_HARD_STOP_S", "ARC_KILL_AT_S",
                                                     "ARC_SERVER_EXTRA_ARGS", "PYTORCH_CUDA_ALLOC_CONF")}, flush=True)


In [ ]:
# Cell 0: clock, paths, fallback submission (before anything that can fail).
import glob, json, os, subprocess, sys, time
T0 = float(os.environ.get("ARC_T0") or time.time()); os.environ["ARC_T0"] = str(T0)
INPUT = os.environ.get("ARC_INPUT_ROOT", "/kaggle/input")
WORK = os.environ.get("ARC_OUT", "/kaggle/working"); os.environ["ARC_OUT"] = WORK
SCRATCH = os.environ.get("ARC_SCRATCH", "/tmp/dsv41"); os.makedirs(SCRATCH, exist_ok=True); os.makedirs(WORK, exist_ok=True)
RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
# Commit runs ("Save & Run All") only check the pipeline: "fallback" writes the fallback and stops; "smoke" runs
# ARC_TASK_LIMIT tasks with short deadlines. The competition rerun always runs "submit".
COMMIT_MODE = os.environ.get("ARC_COMMIT_MODE", "fallback")
MODE = "submit" if RERUN else os.environ.get("ARC_MODE", COMMIT_MODE)

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

# Every cache and HOME under SCRATCH, overriding the image's settings: with a model attached, /root/.cache is
# read-only on Kaggle (smoke v2 and bench v2: uv "Read-only file system" in /root/.cache/uv; setdefault kept the
# image's UV_CACHE_DIR). HOME is changed only for the subprocesses (install, solver, server), not this kernel.
_C = f"{SCRATCH}/cache"
SUB_HOME = f"{SCRATCH}/home"
os.makedirs(SUB_HOME, exist_ok=True)
for _k, _v in (("UV_CACHE_DIR", f"{_C}/uv"), ("XDG_CACHE_HOME", _C), ("VLLM_CACHE_ROOT", f"{_C}/vllm"),
               ("TRITON_CACHE_DIR", f"{_C}/triton"), ("TORCHINDUCTOR_CACHE_DIR", f"{_C}/inductor"),
               ("FLASHINFER_WORKSPACE_BASE", _C), ("TORCH_EXTENSIONS_DIR", f"{_C}/torch_ext"),
               ("HF_HOME", f"{_C}/hf"), ("CUDA_CACHE_PATH", f"{_C}/nv")):
    os.environ[_k] = _v
    os.makedirs(_v, exist_ok=True)
os.environ.pop("UV_NO_CACHE", None)
CHALLENGES = os.environ.get("ARC_CHALLENGES") or find_one(f"{INPUT}/**/arc-agi_test_challenges.json")
os.environ["ARC_CHALLENGES"] = CHALLENGES
_tasks = json.load(open(CHALLENGES))
_fb = {t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in v["test"]] for t, v in _tasks.items()}
with open(f"{WORK}/submission.json.tmp", "w") as f:
    json.dump(_fb, f)
os.replace(f"{WORK}/submission.json.tmp", f"{WORK}/submission.json")
print(f"mode {MODE}; fallback submission for {len(_tasks)} tasks written at +{time.time() - T0:.0f}s", flush=True)
# The kit dataset is one tarball (Kaggle datasets do not keep nested folders reliably): unpack and check sha256.
KIT = os.environ.get("ARC_KIT")
if not KIT:
    try:
        import hashlib, tarfile
        _tgz = find_one(f"{INPUT}/**/dsv41_kit.tgz")
        KIT = f"{SCRATCH}/kit"
        with tarfile.open(_tgz) as tf:
            tf.extractall(KIT)
        _man = json.load(open(f"{KIT}/MANIFEST.json"))
        _bad = [f for f, h in _man.items() if hashlib.sha256(open(f"{KIT}/{f}", "rb").read()).hexdigest() != h]
        assert not _bad, f"kit files differ from MANIFEST: {_bad[:5]}"
        print(f"kit unpacked from {_tgz}: {len(_man)} files, sha256 ok", flush=True)
    except Exception as exc:  # without the kit nothing else can run: keep the fallback submission
        print("kit unavailable, fallback only:", type(exc).__name__, exc, flush=True)
        MODE = "fallback"
MODEL_SRC = os.environ.get("ARC_MODEL_SRC") or (os.path.dirname(find_one(f"{INPUT}/**/e043_manifest.json") or "") or None)
print("kit", KIT, "| model", MODEL_SRC, flush=True)

In [ ]:
# Cell 0b (frozen inputs from the attached pins dataset). The candidate's pinned solver, harness, helper modules and
# vLLM overlay come from a small private dataset (dsv41_pins.tgz + SHA256SUMS) built by
# `build_frozen_mt.py --pins-dataset`; the kit and model inputs stay Version 5's. After cell 0 has unpacked and checked
# the kit, this cell backs up the kit files the pins replace, installs the pins, and checks hashes, required features
# and settings.
# Commit run ("Save & Run All"): any failure RAISES, so the commit errors and the log shows "E046 CHECK FAILED".
# Competition rerun: the same failure restores the kit's v3 files and runs per pair at 36 workers (a scored submission
# beats an error); the rerun path never raises (S10). Uses cell 0's KIT, MODE, RERUN, INPUT and find_one.
import hashlib
import json
import os
import shutil
import tarfile

PINS = json.loads(r"""{"archive": "dsv41_pins.tgz", "archive_sha": "bab63efe6c2943376bd6e54391b98602ff2e07e0f0079305d0bce2d2205e7637", "dataset": "dsv41-candbp-pins", "expect": {"ARC_ADMIT_MAX_WAITING": "1", "ARC_AUTO_REPAIR": "1", "ARC_CONTEXT_LIMIT": "163840", "ARC_KV_ADMIT_MAX": "0.85", "ARC_MULTI_BUDGET": "0.5", "ARC_MULTI_TEST": "1", "ARC_WORKERS": "64"}, "fallback_drop": ["ARC_MULTI_TEST", "ARC_MULTI_BUDGET", "ARC_KV_ADMIT_MAX", "ARC_CONTEXT_LIMIT", "ARC_AUTO_REPAIR", "ARC_BEST_GUESS_AT", "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_REASONING_EFFORT", "ARC_ADMIT_MAX_WAITING"], "fallback_server_extra": "[\"--linear-backend\", \"humming\", \"--compilation-config\", \"{\\\"cudagraph_mode\\\": \\\"FULL\\\", \\\"max_cudagraph_capture_size\\\": 64}\"]", "harness_need": ["--multi-test", "--multi-budget", "--context-limit", "class ContextFull", "import builtins as _b", "def answer_items", "def context_budget"], "new": ["repo/evals/auto_repair.py", "repo/evals/auto_repair_names.json"], "opt_need": {"repo/evals/run_deepseek_tools.py": ["--auto-repair"], "solver_dsv41.py": ["ARC_ADMIT_MAX_WAITING"]}, "sha": {"repo/evals/auto_repair.py": "2292369c4ad26fea4e2b3dc5e82dad31d1180192510484a4a1021a7e89b8a33d", "repo/evals/auto_repair_names.json": "571f47eb1bd077358d2153a943d0e6710c4ec18cf5828174d86c8be1fac03e92", "repo/evals/run_deepseek_tools.py": "882c817f2c93dd8e9ebf65c241a694549c5b7679cbacca5d16906a28dd62eecc", "repo/evals/train_check.py": "8fe361752010bba7599b24d83c4491ee462a2cffb1d9b1d018c67b90901fc982", "solver_dsv41.py": "77fe80a31528eb1e515cc591975fc3dc0cb3d6ee6f7c67a0341df188216e6c05", "vllm_overlay_rw.sha256": "c881136cc57a77bc3fc0988322ef23e26e5f75b846e8f0d2ca33034c7a6202a2", "vllm_overlay_rw.tgz": "be8a8b95ea5a86649a8934b6bfd5ddc79b3b8b3d95995f86f3d19aca0849a088"}, "solver_need": ["MULTI_TEST", "MULTI_BUDGET", "CONTEXT_LIMIT", "KV_ADMIT_MAX", "split_multi", "restart_allowed", "shutdown_socket"]}""")  # filled in by build_frozen_mt.py: hashes, expected settings, feature checks


def pins_sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


def pins_install(kit, archive, pins, state):
    """Back up each kit file the pins replace (files new to the kit have no v3 original), then extract the pins over
    the kit. `state` records the backups as they are made and whether extraction started, so a failure part-way
    leaves the fallback exactly what it needs (S10: a backup counts only once its copy exists)."""
    for member in pins["sha"]:
        if member not in pins["new"]:
            shutil.copy2(f"{kit}/{member}", f"{kit}/{member}.v3orig")
            state["backups"][member] = f"{kit}/{member}.v3orig"
    state["extracted"] = True
    with tarfile.open(archive) as tf:
        tf.extractall(kit)


def pins_check(kit, pins):
    """Raise AssertionError at the first mismatch: member hashes, the features every candidate's solver and harness
    need, the features its opt-in settings rely on, and the settings themselves."""
    for member, sha in pins["sha"].items():
        assert pins_sha256(f"{kit}/{member}") == sha, f"pinned file mismatch: {member}"
    with open(f"{kit}/solver_dsv41.py") as f:
        solver = f.read()
    with open(f"{kit}/repo/evals/run_deepseek_tools.py") as f:
        harness = f.read()
    for need in pins["solver_need"]:
        assert need in solver, f"solver lacks {need}"
    for need in pins["harness_need"]:
        assert need in harness, f"harness lacks {need}"
    for member, needs in pins["opt_need"].items():
        with open(f"{kit}/{member}") as f:
            text = f.read()
        for need in needs:
            assert need in text, f"{member} lacks {need}"
    for key, value in pins["expect"].items():
        assert os.environ.get(key) == value, f"{key}={os.environ.get(key)!r}, expected {value!r}"


def pins_fallback(kit, state, pins):
    """Competition rerun after a failed check: the kit's v3 files back (when the pins were extracted at all), Version
    5's settings (per pair, 36 workers, its server flags, every opt-in lever dropped). Never raises; returns what could
    not be restored."""
    kept = []
    try:
        if state["extracted"]:
            for member in pins["sha"]:
                if member in pins["new"]:
                    continue  # new to the kit: the v3 code never imports it
                try:
                    shutil.copy2(state["backups"][member], f"{kit}/{member}.restore")
                    os.replace(f"{kit}/{member}.restore", f"{kit}/{member}")
                except Exception as exc:
                    kept.append(f"{member} ({type(exc).__name__})")
        # (a pinned solver kept by a failed restore would pass the opt-in harness flags to the v3 harness, which exits)
        for key in pins["fallback_drop"]:
            os.environ.pop(key, None)
        os.environ["ARC_WORKERS"] = "36"
        os.environ["ARC_SERVER_EXTRA_ARGS"] = pins["fallback_server_extra"]  # no flag that needs the new overlay
    except Exception as exc:
        kept.append(f"{type(exc).__name__}: {exc}")
    return kept


if MODE != "fallback":
    _pins_state, _pins_why = {"backups": {}, "extracted": False}, ""
    try:
        assert KIT, "kit not unpacked"
        _pins_archive = find_one(f"{INPUT}/**/{PINS['archive']}")
        assert _pins_archive, f"{PINS['archive']} is not attached (dataset {PINS['dataset']})"
        assert pins_sha256(_pins_archive) == PINS["archive_sha"], f"{_pins_archive} is not the build's archive"
        pins_install(KIT, _pins_archive, PINS, _pins_state)
        pins_check(KIT, PINS)
    except Exception as exc:
        _pins_why = f"{type(exc).__name__}: {exc}"
    if not _pins_why:
        print("E046 CHECK OK: pinned files installed and checked:", {m: h[:8] for m, h in PINS["sha"].items()},
              PINS["expect"], flush=True)
    else:
        print("=" * 100, f"E046 CHECK FAILED: {_pins_why}", "=" * 100, sep="\n", flush=True)
        if not RERUN:
            raise RuntimeError(f"E046 CHECK FAILED (commit run): {_pins_why}")
        _pins_kept = pins_fallback(KIT, _pins_state, PINS)
        print("E046 FALLBACK: the kit's v3 code, per pair, 36 workers"
              + (f"; not restored, kept as is: {', '.join(_pins_kept)}" if _pins_kept else ""), flush=True)


In [ ]:
# Cell 1: stage the checkpoint to local disk in the background (stage_model.py), while vLLM installs.
# "warm" (read the mount once, no writes): smoke v2 (2026-10-24) with "copy" hit "Read-only file system" on the
# root overlay during the install while 131 GB were being written to /tmp.
STAGE_MODE = os.environ.get("ARC_STAGE_MODE", "warm")
STAGE_DONE = f"{SCRATCH}/stage_done"
if MODE != "fallback" and MODEL_SRC:
  try:
    if os.path.exists(STAGE_DONE):
        os.remove(STAGE_DONE)
    _stage = subprocess.Popen([sys.executable, "-I", f"{KIT}/stage_model.py", MODEL_SRC, f"{SCRATCH}/model", STAGE_DONE,
                               STAGE_MODE, os.environ.get("ARC_STAGE_THREADS", "16")],
                              stdout=open(f"{SCRATCH}/stage.log", "w"), stderr=subprocess.STDOUT,
                              stdin=subprocess.DEVNULL, start_new_session=True)
    print(f"staging ({STAGE_MODE}) started, pid {_stage.pid}", flush=True)
  except Exception as exc:  # the solver then loads from MODEL_SRC (no staging marker)
    STAGE_DONE = ""
    print("staging failed to start:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 2: offline install of the fork wheel set, the vLLM overlay, the CUDA 13 shim (E042's Kaggle recipe).
import hashlib, shutil, tarfile
V = os.environ.get("ARC_VENV", f"{SCRATCH}/venv"); PY = f"{V}/bin/python"; SP = f"{V}/lib/python3.12/site-packages"
BASE_ENV = dict(os.environ)
def sh(cmd, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, errors="replace", timeout=timeout, env=env)
    return r.returncode, (r.stdout or "") + (r.stderr or "")
if MODE != "fallback" and not os.environ.get("ARC_SKIP_INSTALL"):
  try:
    real = "/usr/local/nvidia/lib64"  # host driver libcuda first (the image also has an older compat libcuda)
    if os.path.exists(real + "/libcuda.so"):
        os.environ["LD_LIBRARY_PATH"] = real + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_PRELOAD"] = real + "/libcuda.so"
    WHEELS = os.path.dirname(find_one(f"{INPUT}/**/vllm-0.28.1rc1*.whl"))
    t = time.time()
    _uvc = os.environ["UV_CACHE_DIR"]
    _ienv = dict(os.environ, HOME=SUB_HOME, UV_CACHE_DIR=_uvc, TMPDIR=f"{SCRATCH}/tmp")
    os.makedirs(f"{SCRATCH}/tmp", exist_ok=True)
    if shutil.which("uv"):
        rc, out = sh(f"uv venv --cache-dir {_uvc} --python {sys.executable} {V} && uv pip install --cache-dir {_uvc} "
                     f"--python {PY} --no-index --no-deps --find-links {WHEELS} -r {WHEELS}/requirements.lock",
                     timeout=3600, env=_ienv)
    else:
        rc, out = sh(f"{sys.executable} -m venv {V} && {PY} -m pip install --no-cache-dir --no-index --no-deps "
                     f"--find-links {WHEELS} -r {WHEELS}/requirements.lock", timeout=3600, env=_ienv)
    print(f"offline install rc {rc} in {time.time() - t:.0f}s", out[-800:], flush=True)
    assert rc == 0, "vLLM install failed"
    with tarfile.open(f"{KIT}/vllm_overlay_rw.tgz") as tf:
        tf.extractall(SP)
    for line in open(f"{KIT}/vllm_overlay_rw.sha256"):
        h, rel = line.split()
        assert hashlib.sha256(open(f"{SP}/{rel}", "rb").read()).hexdigest() == h, f"overlay mismatch: {rel}"
    print("vLLM overlay installed and checked", flush=True)
    cu = f"{SP}/nvidia/cu13"; shim = f"{SCRATCH}/cuda13"
    os.makedirs(f"{shim}/lib64/stubs", exist_ok=True)
    for d in ("bin", "include", "nvvm", "lib", "share"):
        if os.path.exists(f"{cu}/{d}") and not os.path.lexists(f"{shim}/{d}"):
            os.symlink(f"{cu}/{d}", f"{shim}/{d}")
    for f in os.listdir(f"{cu}/lib"):
        if not os.path.lexists(f"{shim}/lib64/{f}"):
            os.symlink(f"{cu}/lib/{f}", f"{shim}/lib64/{f}")
    for name in ("cudart", "nvrtc"):
        r = sorted(glob.glob(f"{cu}/lib/lib{name}.so.*"))
        if r and not os.path.lexists(f"{shim}/lib64/lib{name}.so"):
            os.symlink(r[0], f"{shim}/lib64/lib{name}.so")
    drv = next(iter(sorted(glob.glob("/usr/local/nvidia/lib64/libcuda.so*")) + sorted(glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*"))), None)
    for dst in (f"{shim}/lib64/stubs/libcuda.so", f"{shim}/lib64/libcuda.so"):
        if drv and not os.path.lexists(dst):
            os.symlink(drv, dst)
    BASE_ENV = dict(os.environ)
    BASE_ENV.update(CUDA_HOME=shim, CUDA_PATH=shim, PATH=f"{V}/bin:{shim}/bin:" + os.environ["PATH"],
                    LD_LIBRARY_PATH=os.environ.get("LD_LIBRARY_PATH", "") + f":{shim}/lib64")
    rc, out = sh(f'nvcc --version | tail -1; {PY} -c "import torch, vllm, flashinfer, importlib.metadata as m; '
                 f'print(torch.__version__, torch.cuda.device_count(), vllm.__version__, flashinfer.__version__, '
                 f'm.version(\'humming-kernels\'))"', timeout=900, env=BASE_ENV)
    print("toolchain / imports:", rc, out[-600:], flush=True)
    rc, out = sh("free -g | head -2; df -h /tmp /kaggle/working | tail -2; nproc; cat /sys/fs/cgroup/memory.max "
                 "/sys/fs/cgroup/memory.oom.group 2>/dev/null; grep -m1 'model name' /proc/cpuinfo", timeout=60)
    print(out, flush=True)
  except Exception as exc:  # no working vLLM: keep the fallback submission
    print("install failed, fallback only:", type(exc).__name__, exc, flush=True)
    MODE = "fallback"
# The tool code runs on the image's Python (scipy etc.); if that cannot import numpy/scipy in the executor's
# environment (-I, PATH only), use the venv's Python instead (numpy only).
TOOL_PY = os.environ.get("ARC_TOOL_PYTHON", sys.executable)
if MODE != "fallback":
    try:
        r = subprocess.run([TOOL_PY, "-I", "-c", "import numpy, scipy.ndimage; print(numpy.__version__)"],
                           env={"PATH": os.environ.get("PATH", ""), "OMP_NUM_THREADS": "1"}, capture_output=True,
                           text=True, timeout=300)
        print("tool python", TOOL_PY, "->", r.returncode, (r.stdout + r.stderr).strip()[-300:], flush=True)
        if r.returncode != 0 and os.path.exists(PY):
            TOOL_PY = PY
    except Exception as exc:
        print("tool python check failed:", exc, flush=True)
BASE_ENV.pop("DSV41_SYNTH_INIT", None)  # the overlay's synthetic init must stay off for real weights
BASE_ENV["HOME"] = SUB_HOME  # /root/.cache is read-only on Kaggle with a model attached
# FlashInfer's sampling module cannot JIT on the Kaggle image (E042 K-runs); vLLM's PyTorch sampler instead.
BASE_ENV["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

In [ ]:
# Cell 3: the solver (vLLM server, tool harness, deadlines, watchdog, incremental submission).
if MODE != "fallback" and MODEL_SRC:
  try:
    import signal, threading
    env = dict(BASE_ENV)
    env.update(ARC_KIT=KIT, ARC_MODEL_DIR=MODEL_SRC, ARC_STAGE_DONE=STAGE_DONE, ARC_VLLM=f"{V}/bin/vllm",
               ARC_TOOL_PYTHON=TOOL_PY, ARC_SCRATCH=SCRATCH)
    if MODE == "smoke":
        env.setdefault("ARC_TASK_LIMIT", "8")
        for k, v in (("ARC_STOP_NEW_S", "4800"), ("ARC_FORCE_AT_S", "5400"), ("ARC_HARD_STOP_S", "5700")):
            env.setdefault(k, v)
    p = subprocess.Popen([PY, "-u", f"{KIT}/solver_dsv41.py"], env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, errors="replace", start_new_session=True)
    # Independent kill switch: whatever hangs inside the solver, its whole session dies at T0 + ARC_KILL_AT_S
    # (default 11h45; the solver's own hard stop is 11h30), so cell 4 and the notebook end before Kaggle's 12 h.
    def _kill():
        try:
            os.killpg(p.pid, signal.SIGKILL)
            print("kill switch: solver session killed", flush=True)
        except Exception:
            pass
    _timer = threading.Timer(max(0.0, T0 + float(os.environ.get("ARC_KILL_AT_S", 42300)) - time.time()), _kill)
    _timer.daemon = True
    _timer.start()
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    _timer.cancel()
    print("solver exit code", p.returncode, flush=True)
  except Exception as exc:
    print("solver cell failed:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 4: final check of submission.json; log archive.
import tarfile
def _ok(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) and 0 < len(r) <= 30 for r in g)
            and len({len(r) for r in g}) == 1 and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))
try:
    sub = json.load(open(f"{WORK}/submission.json"))
    assert set(sub) == set(_tasks), f"task ids differ ({len(sub)} vs {len(_tasks)})"
    for t, v in _tasks.items():
        assert len(sub[t]) == len(v["test"]), f"{t}: {len(sub[t])} rows for {len(v['test'])} test inputs"
        assert all(_ok(r["attempt_1"]) and _ok(r["attempt_2"]) for r in sub[t]), f"{t}: invalid grid"
    print("submission.json valid", flush=True)
except Exception as exc:
    print("submission.json invalid, writing the fallback:", exc, flush=True)
    json.dump(_fb, open(f"{WORK}/submission.json", "w"))
try:
    logs = f"{WORK}/dsv41_logs"
    if os.path.isdir(f"{logs}/traces"):
        with tarfile.open(f"{WORK}/dsv41_traces.tar.gz", "w:gz") as tf:
            tf.add(f"{logs}/traces", arcname="traces")
        shutil.rmtree(f"{logs}/traces", ignore_errors=True)
    for f in ("stage.log",):
        if os.path.exists(f"{SCRATCH}/{f}"):
            shutil.copy(f"{SCRATCH}/{f}", f"{logs}/{f}" if os.path.isdir(logs) else f"{WORK}/{f}")
except Exception as exc:
    print("log archive failed:", exc, flush=True)
print(f"done at +{(time.time() - T0) / 3600:.2f}h", flush=True)

In [ ]:
# Cell 5 (smoke report, commit runs only): peaks of host memory (cgroup, the 173 GiB limit), GPU power / clocks /
# memory / temperature and the lowest free disk, from the sampler started in the settings cell (every 30 s)
import json, os
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN") and os.environ.get("ARC_SMOKE_ENV_LOG"):
    try:
        _recs = [json.loads(l) for l in open(os.environ["ARC_SMOKE_ENV_LOG"]) if l.strip()]
        _gib = lambda v: round(int(v) / 2**30, 1) if v not in (None, "", "max") and str(v).isdigit() else v
        _mx = lambda xs: max([x for x in xs if isinstance(x, (int, float))], default=None)
        print(f"smoke report: {len(_recs)} samples over {(_recs[-1]['t'] - _recs[0]['t']) / 60:.1f} min", flush=True)
        print("  cgroup memory.max", _gib(_recs[-1].get("memory.max")), "GiB; peak memory.current",
              _mx([_gib(r.get("memory.current")) for r in _recs]), "GiB; memory.peak", _gib(_recs[-1].get("memory.peak")),
              "GiB; swap", _gib(_recs[-1].get("memory.swap.current")), flush=True)
        for k in ("anon", "shmem", "file", "unevictable"):
            print(f"  peak cgroup {k}", _mx([_gib(r.get("stat", {}).get(k)) for r in _recs]), "GiB", flush=True)
        _mi = [r.get("meminfo_kb", {}) for r in _recs]
        print("  host MemTotal", round(_mi[-1].get("MemTotal", 0) / 2**20, 1), "GiB; min MemAvailable",
              round(min(m.get("MemAvailable", 0) for m in _mi) / 2**20, 1), "GiB; peak Shmem",
              round(max(m.get("Shmem", 0) for m in _mi) / 2**20, 1), "GiB; peak Mlocked",
              round(max(m.get("Mlocked", 0) for m in _mi) / 2**20, 1), "GiB", flush=True)
        _g = {}
        for r in _recs:
            for line in r.get("gpu", []):
                f = [x.strip() for x in line.split(",")]
                if len(f) == 7:
                    d = _g.setdefault(f[0], {"draw": [], "limit": f[2], "sm": [], "mem": [], "total": f[5], "temp": []})
                    for k, i in (("draw", 1), ("sm", 3), ("mem", 4), ("temp", 6)):
                        try:
                            d[k].append(float(f[i]))
                        except ValueError:
                            pass
        for i, d in sorted(_g.items()):
            print(f"  GPU {i}: power limit {d['limit']} W, peak draw {_mx(d['draw'])} W, peak SM {_mx(d['sm'])} MHz, "
                  f"peak memory {_mx(d['mem'])} / {d['total']} MiB, peak {_mx(d['temp'])} C", flush=True)
        _df = {}
        for r in _recs:
            for k, v in r.get("disk_free_gib", {}).items():
                _df[k] = min(_df.get(k, v), v)
        print("  lowest free disk (GiB):", _df, flush=True)
    except Exception as exc:
        print("smoke report failed:", type(exc).__name__, exc, flush=True)
